# Treino de pose — MindTrace, 6 pontos

Compara **CSPNeXt-S** (leve, para CPU) e **ResNet-50** (referencia) sobre a mesma base rotulada.

A rotulagem já está feita (287 quadros úteis, 33 arenas, 24 animais) e é compartilhada pelos
dois backbones — por isso comparar custa apenas tempo de GPU, que aqui é gratuito.

**Antes de começar:** suba a pasta do projeto DLC para o seu Google Drive.
Não é preciso subir os vídeos: o treino usa apenas `labeled-data/` e o `config.yaml`,
cerca de 20 MB contra 363 MB de vídeo.

**Ative a GPU:** Ambiente de execução → Alterar tipo de ambiente de execução → T4 GPU.

## 1. Confirmar que há GPU

In [ ]:
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print(out if out else "SEM GPU — troque o tipo de ambiente antes de seguir")

## 2. Instalar o DeepLabCut

Demora alguns minutos.

**O runtime vai reiniciar sozinho ao final — isso é esperado, não é erro.** O DLC exige
`numpy<2`, e o Colab já veio com numpy 2.x carregado em memória; sem reiniciar, o
`import deeplabcut` quebra com `ValueError` porque encontra extensões compiladas contra
a versão errada.

Os avisos vermelhos de conflito de dependência (`jax`, `cupy`, `rasterio`…) são esperados
e inofensivos: são pacotes do Colab que queriam numpy 2, e nada disso é usado pelo treino.

Depois do reinício, **siga da célula 3** — não repita esta.

In [ ]:
!pip install -q deeplabcut==3.0.1

import IPython
print("instalado; reiniciando o runtime para carregar o numpy correto...")
IPython.Application.instance().kernel.do_shutdown(True)

### 2b. Conferir a instalação

Rode esta **depois** que o runtime reiniciar.

In [ ]:
import deeplabcut
import numpy
print("DeepLabCut", deeplabcut.__version__)
print("numpy", numpy.__version__, "(precisa ser 1.x)")

## 3. Montar o Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 4. Apontar para o projeto e corrigir os caminhos

O `config.yaml` foi gravado no Windows e guarda caminhos absolutos daquela máquina, que aqui
não existem. Esta célula os reescreve para o Drive — é a única fricção real da transferência.

**Ajuste `PROJECT`** para onde você colocou a pasta.

In [ ]:
from pathlib import Path
import ruamel.yaml as yaml_mod

PROJECT = Path("/content/drive/MyDrive/mindtrace-pose-memorylab-2026-09-08")
assert PROJECT.is_dir(), f"nao encontrei {PROJECT} — confira o caminho no Drive"

config_path = PROJECT / "config.yaml"
yaml = yaml_mod.YAML()
cfg = yaml.load(config_path.read_text(encoding="utf-8"))

cfg["project_path"] = str(PROJECT)

# Descarta vídeos sem pasta de rótulos — arenas vazias removidas durante a
# rotulagem continuam listadas no config e fariam o create_training_dataset falhar.
labeled = PROJECT / "labeled-data"
kept, dropped = {}, []
for key, value in dict(cfg.get("video_sets") or {}).items():
    name = Path(str(key).replace(chr(92), "/")).stem
    if (labeled / name).is_dir():
        kept[str(PROJECT / "videos" / (name + ".mp4"))] = value
    else:
        dropped.append(name)

cfg["video_sets"] = kept
with open(config_path, "w", encoding="utf-8") as fh:
    yaml.dump(cfg, fh)

print("project_path:", cfg["project_path"])
print("videos mantidos:", len(kept))
if dropped:
    print("descartados (sem pasta de rotulos):", dropped)
print("pontos:", list(cfg["bodyparts"]))

## 5. Conferir a base rotulada

Confirme que o upload trouxe tudo antes de gastar horas de GPU.

In [ ]:
import pandas as pd

labeled = PROJECT / "labeled-data"
folders = sorted(d for d in labeled.iterdir() if d.is_dir() and not d.name.endswith("_labeled"))
n_kp = len(cfg["bodyparts"])
total = complete = partial = empty = 0

for folder in folders:
    images = list(folder.glob("*.png"))
    files = list(folder.glob("CollectedData_*.h5")) or list(folder.glob("CollectedData_*.csv"))
    total += len(images)
    if not files:
        empty += len(images)
        continue
    if files[0].suffix == ".h5":
        df = pd.read_hdf(files[0])
    else:
        df = pd.read_csv(files[0], header=[0, 1, 2], index_col=[0, 1, 2])
    counts = (~df.isna()).sum(axis=1) // 2
    complete += int((counts == n_kp).sum())
    partial += int(((counts > 0) & (counts < n_kp)).sum())
    empty += int((counts == 0).sum())

print(f"pastas: {len(folders)}  imagens: {total}")
print(f"completos: {complete}   parciais: {partial}   vazios: {empty}")
print(f"utilizaveis no treino: {complete + partial}")

## 6. Criar os dois conjuntos de treino

Mesmos dados, backbones diferentes.

O motor PyTorch do DLC 3.x nao oferece MobileNet — aquele era o backend TensorFlow antigo.
O substituto e o **cspnext_s**, da familia RTMPose, desenhado para inferencia rapida em CPU.
As demais opcoes do enum (`dekr_*`, `ctd_coam_*`) resolvem multi-animal e nao se aplicam aqui.

`userfeedback=False` sobrescreve shuffles existentes — torna a celula re-executavel se uma
tentativa anterior tiver deixado um shuffle pela metade.

In [ ]:
deeplabcut.create_training_dataset(str(config_path), Shuffles=[1], net_type="cspnext_s", userfeedback=False)
deeplabcut.create_training_dataset(str(config_path), Shuffles=[2], net_type="resnet_50", userfeedback=False)
print("shuffle 1 = cspnext_s (leve) | shuffle 2 = resnet_50 (referencia)")

## 7. Treinar

Cada um leva algumas horas numa T4. O Colab gratuito derruba a sessão por inatividade —
deixe a aba aberta. Se cair, o DLC retoma do ultimo checkpoint: basta rodar a célula de novo.

In [ ]:
import time
t0 = time.time()
deeplabcut.train_network(str(config_path), shuffle=1, displayiters=500, saveiters=5000, maxiters=150000)
print(f"CSPNeXt-S treinado em {(time.time() - t0) / 3600:.1f} h")

In [ ]:
import time
t0 = time.time()
deeplabcut.train_network(str(config_path), shuffle=2, displayiters=500, saveiters=5000, maxiters=150000)
print(f"ResNet-50 treinado em {(time.time() - t0) / 3600:.1f} h")

## 8. Avaliar

`evaluate_network` devolve o erro em pixels no conjunto de teste — a métrica que o modelo
de pose atual do laboratório nunca teve.

**Compare contra o seu proprio ruido de clique, não contra zero.** Se você rotular 20 quadros
de novo às cegas e variar 4 px, então 4 px é o teto: nenhum modelo pode ser significativamente
melhor que o alvo que você deu a ele.

In [ ]:
import glob
import pandas as pd

deeplabcut.evaluate_network(str(config_path), Shuffles=[1, 2], plotting=False)

for path in glob.glob(str(PROJECT / "evaluation-results*" / "**" / "*.csv"), recursive=True):
    print("\n==", Path(path).name, "==")
    print(pd.read_csv(path).to_string(index=False))

## 9. Exportar para ONNX

O pipeline Python e o app Qt consomem ONNX. O DLC 3.x é PyTorch, então a saída provavelmente
vem em **NCHW**, enquanto o modelo atual do MindTrace é **NHWC**. A decodificação precisa ser
ajustada dos dois lados, e o sintoma de não ajustar são coordenadas absurdas, não um erro.

**Anote a forma de saída impressa abaixo** — é ela que define o ajuste.

In [ ]:
import glob

for shuffle, name in [(1, "cspnext_s"), (2, "resnet50")]:
    try:
        deeplabcut.export_model(str(config_path), shuffle=shuffle, make_tar=False)
        print(f"shuffle {shuffle} ({name}): exportado")
    except Exception as exc:
        print(f"shuffle {shuffle} ({name}): export_model falhou -> {exc}")

print("\nexportados:", glob.glob(str(PROJECT / "exported-models" / "*")))

In [ ]:
# Reserva, caso a célula acima falhe: localizar o snapshot e converter manualmente.
import glob

snapshots = sorted(glob.glob(str(PROJECT / "dlc-models-pytorch" / "**" / "snapshot*.pt"), recursive=True))
print("snapshots encontrados:")
for s in snapshots:
    print("  ", s)
print()
print("carregue o snapshot e rode torch.onnx.export com entrada (1, 3, 240, 360)")
print("ATENCAO: PyTorch usa NCHW (1,3,240,360); o MindTrace atual espera NHWC (1,240,360,3)")

## 10. O que trazer de volta

Baixe do Drive para a máquina de análise:

- o `.onnx` de cada backbone
- os CSVs de `evaluation-results`, com o erro em pixels
- o tempo de treino de cada um

Na máquina local, o passo seguinte é medir **tempo de inferência em CPU** dos dois — é o
requisito que decide, já que o laboratório roda sem GPU. O `scripts/extract_pose.py` do
repositório aceita `--model`, então basta apontar para cada `.onnx` e comparar.

A decisão não sai só do erro em pixels: um modelo 1 px melhor que rode 5x mais devagar não
atende ao requisito, e 1 px dificilmente muda se o rato está caminhando ou congelado.